In [ ]:
"""
============================================================================
LSTM MCQ Solver — standalone script (no pretrained models)
============================================================================
Trains a from-scratch, Siamese-style bidirectional LSTM to solve the
Smart MCQ Solver Challenge and produces submission.csv.

Architecture:
  - Shared BiLSTM encoder embeds the prompt and each of the 5 options
    (mean-pooled over non-pad timesteps)
  - A small MLP scoring head combines
        [prompt_vec, option_vec, |prompt-option|, prompt*option]
    into a single compatibility score per option
  - Softmax over the 5 scores -> cross-entropy loss against the true label

Install deps:
    pip install torch scikit-learn pandas numpy wandb matplotlib

W&B: this script assumes you're already logged in (wandb login done once
with your token), or set the WANDB_API_KEY env var before running:
    export WANDB_API_KEY=your_token_here

NOTE: written for review, not executed here — check DATA_DIR before running.
============================================================================
"""

import os
import re
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
SEED = 42
DATA_DIR = "/content/drive/MyDrive/smart-mcq-solver"     # <-- update to wherever train.csv/test.csv live
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")
OUTPUT_DIR = "/content/drive/MyDrive/smart-mcq-solver/output_lstm"
SUBMISSION_PATH = os.path.join(OUTPUT_DIR, "submission.csv")
CONFUSION_MATRIX_PATH = os.path.join(OUTPUT_DIR, "lstm_confusion_matrix.png")

OPTION_COLS = ["A", "B", "C", "D", "E"]
LABEL2IDX = {l: i for i, l in enumerate(OPTION_COLS)}
IDX2LABEL = {i: l for l, i in LABEL2IDX.items()}

USE_WANDB = True
WANDB_PROJECT = "smart-mcq-solver"
WANDB_ENTITY = None          # set to your W&B username/team if needed

# model / training hyperparams
EMBED_DIM = 128
HIDDEN_DIM = 128
NUM_LSTM_LAYERS = 1
MAX_PROMPT_LEN = 60
MAX_OPT_LEN = 25
VAL_FRAC = 0.1
BATCH_SIZE = 32
NUM_EPOCHS = 30
LR = 1e-3
PATIENCE = 4                 # early stopping on val mAP@3
MIN_VOCAB_FREQ = 2
MAX_VOCAB_SIZE = 30000

os.makedirs(OUTPUT_DIR, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


# ============================================================================
# DATA LOADING & CLEANING
# ============================================================================

def load_data(train_csv=TRAIN_CSV, test_csv=TEST_CSV):
    """train.csv: id, prompt, A, B, C, D, E, answer
    test.csv : id, prompt, A, B, C, D, E
    """
    train_df = pd.read_csv(train_csv)
    test_df = pd.read_csv(test_csv)
    return train_df, test_df


def clean_text(text) -> str:
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"<[^>]+>", " ", text)              # strip HTML
    text = re.sub(r"http\S+|www\.\S+", " ", text)      # strip URLs
    text = re.sub(r"\s+", " ", text).strip()
    return text


def handle_missing_data(df: pd.DataFrame, text_cols) -> pd.DataFrame:
    df = df.copy()
    for c in text_cols:
        if c in df.columns:
            df[c] = df[c].apply(clean_text)
    all_missing_mask = (df[text_cols] == "").all(axis=1)
    if all_missing_mask.any():
        print(f"Dropping {all_missing_mask.sum()} fully-empty rows")
        df = df[~all_missing_mask].reset_index(drop=True)
    return df


def tokenize_simple(text: str):
    return re.findall(r"[a-zA-Z0-9']+", text.lower())


# ============================================================================
# METRIC — mAP@3
# ============================================================================

def mapk_score(y_true_labels, y_pred_rankings, k=3):
    """y_true_labels: ['B', 'A', ...] | y_pred_rankings: [['A','B','C'], ...]"""
    scores = []
    for true, preds in zip(y_true_labels, y_pred_rankings):
        preds = preds[:k]
        score = 0.0
        for i, p in enumerate(preds):
            if p == true:
                score = 1.0 / (i + 1)
                break
        scores.append(score)
    return float(np.mean(scores))


def logits_to_top3(logits):
    order = np.argsort(-logits, axis=1)
    return [[IDX2LABEL[j] for j in row[:3]] for row in order]


# ============================================================================
# VOCAB
# ============================================================================

class Vocab:
    """Frequency-based vocabulary built from the train+test corpus (word
    surface forms only — building the vocab from test text does NOT leak
    labels, it just means unseen test words aren't all mapped to <unk>)."""

    def __init__(self, texts, min_freq=MIN_VOCAB_FREQ, max_size=MAX_VOCAB_SIZE):
        counter = Counter()
        for t in texts:
            counter.update(tokenize_simple(t))
        self.itos = ["<pad>", "<unk>"] + [
            w for w, c in counter.most_common(max_size) if c >= min_freq
        ]
        self.stoi = {w: i for i, w in enumerate(self.itos)}
        self.pad_id = self.stoi["<pad>"]
        self.unk_id = self.stoi["<unk>"]

    def encode(self, text, max_len):
        ids = [self.stoi.get(tok, self.unk_id) for tok in tokenize_simple(text)][:max_len]
        if len(ids) < max_len:
            ids = ids + [self.pad_id] * (max_len - len(ids))
        return ids

    def __len__(self):
        return len(self.itos)


# ============================================================================
# DATASET
# ============================================================================

class MCQLstmDataset(Dataset):
    def __init__(self, df, vocab, max_prompt_len=MAX_PROMPT_LEN, max_opt_len=MAX_OPT_LEN, has_labels=True):
        self.df = df.reset_index(drop=True)
        self.vocab = vocab
        self.max_prompt_len = max_prompt_len
        self.max_opt_len = max_opt_len
        self.has_labels = has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt_ids = self.vocab.encode(row["prompt"], self.max_prompt_len)
        option_ids = [self.vocab.encode(row[c], self.max_opt_len) for c in OPTION_COLS]
        item = {
            "prompt_ids": torch.tensor(prompt_ids, dtype=torch.long),
            "option_ids": torch.tensor(option_ids, dtype=torch.long),   # [5, max_opt_len]
        }
        if self.has_labels:
            item["label"] = torch.tensor(LABEL2IDX[row["answer"]], dtype=torch.long)
        return item


def lstm_collate(batch):
    out = {
        "prompt_ids": torch.stack([b["prompt_ids"] for b in batch]),
        "option_ids": torch.stack([b["option_ids"] for b in batch]),
    }
    if "label" in batch[0]:
        out["label"] = torch.stack([b["label"] for b in batch])
    return out


# ============================================================================
# MODEL
# ============================================================================

class LSTMEncoder(nn.Module):
    """Shared bidirectional LSTM encoder, mean-pooled over non-pad timesteps
    (more robust than the final hidden state when sequences are padded)."""

    def __init__(self, vocab_size, embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM,
                 num_layers=NUM_LSTM_LAYERS, dropout=0.3, pad_id=0):
        super().__init__()
        self.pad_id = pad_id
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, num_layers=num_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        mask = (x != self.pad_id).float()          # [batch, seq_len]
        emb = self.embedding(x)
        out, _ = self.lstm(emb)                      # [batch, seq_len, hidden*2]
        lengths = mask.sum(dim=1, keepdim=True).clamp(min=1)
        pooled = (out * mask.unsqueeze(-1)).sum(dim=1) / lengths
        return self.dropout(pooled)                  # [batch, hidden*2]


class MCQLstmModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM, pad_id=0):
        super().__init__()
        self.encoder = LSTMEncoder(vocab_size, embed_dim, hidden_dim, pad_id=pad_id)
        enc_dim = hidden_dim * 2
        self.scorer = nn.Sequential(
            nn.Linear(enc_dim * 4, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, prompt_ids, option_ids):
        # prompt_ids: [batch, plen] | option_ids: [batch, 5, olen]
        batch_size, n_opts, olen = option_ids.shape
        prompt_vec = self.encoder(prompt_ids)                               # [batch, enc_dim]
        opt_vec = self.encoder(option_ids.view(batch_size * n_opts, olen))
        opt_vec = opt_vec.view(batch_size, n_opts, -1)                      # [batch, 5, enc_dim]

        prompt_exp = prompt_vec.unsqueeze(1).expand(-1, n_opts, -1)         # [batch, 5, enc_dim]
        combined = torch.cat([
            prompt_exp, opt_vec,
            torch.abs(prompt_exp - opt_vec),
            prompt_exp * opt_vec,
        ], dim=-1)                                                          # [batch, 5, enc_dim*4]

        return self.scorer(combined).squeeze(-1)                            # [batch, 5] logits


# ============================================================================
# ERROR ANALYSIS — Macro F1 + confusion matrix
# ============================================================================

def run_error_analysis(df, logits, log_to_wandb=False, save_plot=True):
    from sklearn.metrics import f1_score, confusion_matrix

    top1_pred_idx = np.argmax(logits, axis=1)
    pred_labels = [IDX2LABEL[i] for i in top1_pred_idx]
    true_labels = df["answer"].tolist()

    macro_f1 = f1_score(true_labels, pred_labels, average="macro", labels=OPTION_COLS)
    cm = confusion_matrix(true_labels, pred_labels, labels=OPTION_COLS)

    wrong_mask = np.array(pred_labels) != np.array(true_labels)
    wrong_examples = df.loc[wrong_mask, ["id", "prompt", "answer"]].copy()
    wrong_examples["predicted"] = np.array(pred_labels)[wrong_mask]
    wrong_examples = wrong_examples.head(20)

    if save_plot:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(5, 4))
        im = ax.imshow(cm, cmap="Blues")
        ax.set_xticks(range(len(OPTION_COLS)))
        ax.set_yticks(range(len(OPTION_COLS)))
        ax.set_xticklabels(OPTION_COLS)
        ax.set_yticklabels(OPTION_COLS)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        ax.set_title("Confusion Matrix — LSTM")
        for i in range(len(OPTION_COLS)):
            for j in range(len(OPTION_COLS)):
                ax.text(j, i, cm[i, j], ha="center", va="center",
                        color="white" if cm[i, j] > cm.max() / 2 else "black")
        fig.colorbar(im)
        fig.tight_layout()
        fig.savefig(CONFUSION_MATRIX_PATH, dpi=150)
        plt.close(fig)

    if log_to_wandb:
        import wandb
        wandb.log({
            "macro_f1": macro_f1,
            "confusion_matrix_image": wandb.Image(CONFUSION_MATRIX_PATH) if save_plot else None,
            "misclassified_examples": wandb.Table(dataframe=wrong_examples),
        })

    return {
        "macro_f1": macro_f1,
        "confusion_matrix": cm,
        "wrong_examples": wrong_examples,
    }


# ============================================================================
# TRAIN / EVAL LOOP
# ============================================================================

def run_eval(model, loader, device):
    model.eval()
    logits_list, labels_list = [], []
    with torch.no_grad():
        for batch in loader:
            prompt_ids = batch["prompt_ids"].to(device)
            option_ids = batch["option_ids"].to(device)
            logits = model(prompt_ids, option_ids)
            logits_list.append(logits.cpu().numpy())
            if "label" in batch:
                labels_list.extend(batch["label"].numpy().tolist())
    return np.concatenate(logits_list, axis=0), labels_list


def train_lstm(train_df, test_df):
    from sklearn.model_selection import train_test_split

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Training on device: {device}")

    tr_df, val_df = train_test_split(
        train_df, test_size=VAL_FRAC, random_state=SEED, stratify=train_df["answer"]
    )

    # vocab from train+test text (word surface forms only, no label leakage)
    vocab_texts = []
    for df in (train_df, test_df):
        for c in ["prompt"] + OPTION_COLS:
            vocab_texts.extend(df[c].tolist())
    vocab = Vocab(vocab_texts)
    print(f"Vocab size: {len(vocab)}")

    train_ds = MCQLstmDataset(tr_df, vocab)
    val_ds = MCQLstmDataset(val_df, vocab)
    test_ds = MCQLstmDataset(test_df, vocab, has_labels=False)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=lstm_collate)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=lstm_collate)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=lstm_collate)

    model = MCQLstmModel(len(vocab), pad_id=vocab.pad_id).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=1)
    criterion = nn.CrossEntropyLoss()

    if USE_WANDB:
        import wandb
        wandb.init(
            project=WANDB_PROJECT, entity=WANDB_ENTITY, name="lstm-siamese-scratch",
            config={
                "embed_dim": EMBED_DIM, "hidden_dim": HIDDEN_DIM, "lr": LR,
                "num_epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE, "vocab_size": len(vocab),
            },
        )

    best_val_map3 = -1.0
    best_state = None
    epochs_no_improve = 0

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            optimizer.zero_grad()
            prompt_ids = batch["prompt_ids"].to(device)
            option_ids = batch["option_ids"].to(device)
            labels = batch["label"].to(device)

            logits = model(prompt_ids, option_ids)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)  # LSTMs can explode-gradient
            optimizer.step()
            total_loss += loss.item() * prompt_ids.size(0)

        train_loss = total_loss / len(train_ds)

        val_logits, val_labels_idx = run_eval(model, val_loader, device)
        val_true = [IDX2LABEL[l] for l in val_labels_idx]
        val_rankings = logits_to_top3(val_logits)
        val_map3 = mapk_score(val_true, val_rankings)
        scheduler.step(val_map3)

        print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_map3={val_map3:.4f}")
        if USE_WANDB:
            import wandb
            wandb.log({
                "epoch": epoch, "train_loss": train_loss, "val_map3": val_map3,
                "lr": optimizer.param_groups[0]["lr"],
            })

        if val_map3 > best_val_map3:
            best_val_map3 = val_map3
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f"Early stopping at epoch {epoch} (no val improvement for {PATIENCE} epochs)")
                break

    model.load_state_dict(best_state)
    model.to(device)

    # ---- final validation metrics (Macro F1, confusion matrix) --------------
    final_val_logits, _ = run_eval(model, val_loader, device)
    error_metrics = run_error_analysis(val_df, final_val_logits, log_to_wandb=USE_WANDB)
    print(f"Best val mAP@3: {best_val_map3:.4f} | Macro F1: {error_metrics['macro_f1']:.4f}")

    if USE_WANDB:
        import wandb
        wandb.log({"final_val_map3": best_val_map3, "final_val_macro_f1": error_metrics["macro_f1"]})
        wandb.finish()

    # ---- test-set predictions -> submission.csv ------------------------------
    test_logits, _ = run_eval(model, test_loader, device)
    test_rankings = logits_to_top3(test_logits)

    return model, vocab, best_val_map3, error_metrics, test_rankings


def build_submission(test_df, rankings, out_path=SUBMISSION_PATH):
    sub = pd.DataFrame({
        "ID": test_df["id"].values,
        "Prediction": [" ".join(r[:3]) for r in rankings],
    })
    sub.to_csv(out_path, index=False)
    print(f"Saved submission to {out_path}")
    return sub


In [ ]:
# ============================================================================
# MAIN
# ============================================================================

def main():
    train_df, test_df = load_data()
    train_df = handle_missing_data(train_df, ["prompt"] + OPTION_COLS)
    test_df = handle_missing_data(test_df, ["prompt"] + OPTION_COLS)

    model, vocab, best_val_map3, error_metrics, test_rankings = train_lstm(train_df, test_df)

    build_submission(test_df, test_rankings)


if __name__ == "__main__":
    main()

Training on device: cpu


wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


Vocab size: 3004
wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 26ds2000018 (26ds2000018-iitmaana) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 01 | train_loss=1.3711 | val_map3=0.8600
Epoch 02 | train_loss=0.3777 | val_map3=0.9483
Epoch 03 | train_loss=0.1782 | val_map3=0.9525
Epoch 04 | train_loss=0.1373 | val_map3=0.9583
Epoch 05 | train_loss=0.1166 | val_map3=0.9583
Epoch 06 | train_loss=0.1142 | val_map3=0.9583
Epoch 07 | train_loss=0.1061 | val_map3=0.9583
Epoch 08 | train_loss=0.1108 | val_map3=0.9583
Early stopping at epoch 8 (no val improvement for 4 epochs)
Best val mAP@3: 0.9583 | Macro F1: 0.9316


epoch,▁▂▃▄▅▆▇█
final_val_macro_f1,▁
final_val_map3,▁
lr,█████▃▃▁
macro_f1,▁
train_loss,█▃▁▁▁▁▁▁
val_map3,▁▇██████
epoch,8
final_val_macro_f1,0.93156
final_val_map3,0.95833
lr,0.00025


Saved submission to /content/drive/MyDrive/smart-mcq-solver/output_lstm/submission.csv


In [ ]:
""" ============================================================================
LSTM MCQ Solver — standalone script (no pretrained models)
============================================================================
Trains a from-scratch, Siamese-style bidirectional LSTM to solve the
Smart MCQ Solver Challenge and produces submission.csv.

This version adds three generalization-focused upgrades over a plain train/val split,
aimed at closing the gap between local validation score and the Kaggle leaderboard score:

1. OPTION-SHUFFLE AUGMENTATION (training time)
   Each epoch, the 5 options are randomly re-ordered (and the label remapped)
   before being fed to the model. This directly prevents the model from learning
   positional shortcuts (e.g. "the answer is usually C") instead of actually
   reading option content — a common cause of a train/val score that doesn't
   transfer to a held-out leaderboard set.

2. K-FOLD CROSS-VALIDATION + BAGGED TEST PREDICTIONS
   Instead of one 90/10 split (whose val score is sensitive to exactly which rows
   land in val), we train N_FOLDS models on different folds and average their
   test-set probabilities. This produces a more trustworthy out-of-fold (OOF)
   validation score AND a more robust, lower-variance final prediction.

3. TEST-TIME AUGMENTATION (TTA) via option shuffling
   At inference, each test question is run through the model multiple times with
   options shuffled into different orders; the resulting per-letter probabilities
   are un-shuffled back to the original A-E order and averaged. This further
   reduces any residual position bias at prediction time, on top of fix #1 during
   training.

Architecture (unchanged): a shared BiLSTM encoder embeds the prompt and each of
the 5 options (mean-pooled over non-pad timesteps); an MLP scoring head combines
[prompt_vec, option_vec, |prompt-option|, prompt*option] into a compatibility
score per option; softmax + cross-entropy training.

Install deps:
    pip install torch scikit-learn pandas numpy wandb matplotlib

W&B: assumes you're already logged in (wandb.login(key=...) or WANDB_API_KEY env var set before running).

NOTE: written for review, not executed here — check DATA_DIR before running.
============================================================================
"""
import os
import re
import random
import warnings
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")


# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
SEED = 42
DATA_DIR = "/content/drive/MyDrive/smart-mcq-solver"  # <-- update to wherever train.csv/test.csv live
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")
OUTPUT_DIR = "/content/drive/MyDrive/smart-mcq-solver/output_lstm_5vfold"
SUBMISSION_PATH = os.path.join(OUTPUT_DIR, "submission_lstm_5v.csv")
CONFUSION_MATRIX_PATH = os.path.join(OUTPUT_DIR, "lstm_confusion_matrix.png")

OPTION_COLS = ["A", "B", "C", "D", "E"]
LABEL2IDX = {l: i for i, l in enumerate(OPTION_COLS)}
IDX2LABEL = {i: l for l, i in LABEL2IDX.items()}

USE_WANDB = True
WANDB_PROJECT = "smart-mcq-solver"
WANDB_ENTITY = None  # set to your W&B username/team if needed

# model / training hyperparams
EMBED_DIM = 128
HIDDEN_DIM = 128
NUM_LSTM_LAYERS = 1
MAX_PROMPT_LEN = 60
MAX_OPT_LEN = 25
BATCH_SIZE = 32
NUM_EPOCHS = 30
LR = 1e-3
PATIENCE = 4  # early stopping on val mAP@3
MIN_VOCAB_FREQ = 2
MAX_VOCAB_SIZE = 30000

# generalization upgrades
SHUFFLE_OPTIONS_TRAINING = True  # augmentation #1
N_FOLDS = 5  # k-fold CV + bagging #2
TTA_AUGMENTS = 5  # test-time augmentation #3 (1 = no TTA, just original order)

os.makedirs(OUTPUT_DIR, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


# ============================================================================
# DATA LOADING & CLEANING
# ============================================================================

def load_data(train_csv=TRAIN_CSV, test_csv=TEST_CSV):
    """train.csv: id, prompt, A, B, C, D, E, answer
    test.csv : id, prompt, A, B, C, D, E
    """
    train_df = pd.read_csv(train_csv)
    test_df = pd.read_csv(test_csv)
    return train_df, test_df


def clean_text(text) -> str:
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"<[^>]+>", " ", text)  # strip HTML
    text = re.sub(r"http\S+|www\.\S+", " ", text)  # strip URLs
    text = re.sub(r"\s+", " ", text).strip()
    return text


def handle_missing_data(df: pd.DataFrame, text_cols) -> pd.DataFrame:
    df = df.copy()
    for c in text_cols:
        if c in df.columns:
            df[c] = df[c].apply(clean_text)
    all_missing_mask = (df[text_cols] == "").all(axis=1)
    if all_missing_mask.any():
        print(f"Dropping {all_missing_mask.sum()} fully-empty rows")
        df = df[~all_missing_mask].reset_index(drop=True)
    return df


def tokenize_simple(text: str):
    return re.findall(r"[a-zA-Z0-9']+", text.lower())


def run_diagnostics(train_df):
    """Quick checks for the shortcut-learning / duplication causes behind a
    train-val vs. leaderboard gap. Purely informational — doesn't change
    behavior, just prints what to look for."""
    print("\n--- Diagnostics ---")
    print("Answer letter distribution:")
    print(train_df["answer"].value_counts(normalize=True).round(3))
    dup_prompts = train_df.duplicated(subset=["prompt"]).sum()
    dup_full = train_df.duplicated(subset=["prompt"] + OPTION_COLS).sum()
    print(f"Duplicate prompts: {dup_prompts} | Fully duplicate rows: {dup_full}")

    def correct_len(row):
        return len(str(row[row["answer"]]))

    avg_correct_len = train_df.apply(correct_len, axis=1).mean()
    avg_option_lens = {c: train_df[c].astype(str).str.len().mean() for c in OPTION_COLS}
    print(f"Avg length of CORRECT option: {avg_correct_len:.1f}")
    print(f"Avg length per option column: {avg_option_lens}")
    print("--- End diagnostics ---\n")


# ============================================================================
# METRIC — mAP@3
# ============================================================================

def mapk_score(y_true_labels, y_pred_rankings, k=3):
    """y_true_labels: ['B', 'A', ...] | y_pred_rankings: [['A','B','C'], ...]"""
    scores = []
    for true, preds in zip(y_true_labels, y_pred_rankings):
        preds = preds[:k]
        score = 0.0
        for i, p in enumerate(preds):
            if p == true:
                score = 1.0 / (i + 1)
                break
        scores.append(score)
    return float(np.mean(scores))


def probs_to_top3(probs):
    order = np.argsort(-probs, axis=1)
    return [[IDX2LABEL[j] for j in row[:3]] for row in order]


# ============================================================================
# VOCAB
# ============================================================================

class Vocab:
    """Frequency-based vocabulary built from the train+test corpus (word
    surface forms only — building the vocab from test text does NOT leak
    labels, it just means unseen test words aren't all mapped to <unk>)."""

    def __init__(self, texts, min_freq=MIN_VOCAB_FREQ, max_size=MAX_VOCAB_SIZE):
        counter = Counter()
        for t in texts:
            counter.update(tokenize_simple(t))
        self.itos = ["<pad>", "<unk>"] + [
            w for w, c in counter.most_common(max_size) if c >= min_freq
        ]
        self.stoi = {w: i for i, w in enumerate(self.itos)}
        self.pad_id = self.stoi["<pad>"]
        self.unk_id = self.stoi["<unk>"]

    def encode(self, text, max_len):
        ids = [self.stoi.get(tok, self.unk_id) for tok in tokenize_simple(text)][:max_len]
        if len(ids) < max_len:
            ids = ids + [self.pad_id] * (max_len - len(ids))
        return ids

    def __len__(self):
        return len(self.itos)


# ============================================================================
# DATASET (with option-shuffle augmentation)
# ============================================================================

class MCQLstmDataset(Dataset):
    """
    shuffle_options=True: each call to __getitem__ randomly re-orders the 5
    options (and remaps the label to match). This is what should be turned ON
    for the training split and OFF for val/test (val/test order should stay
    fixed and comparable across epochs/models; test-time augmentation is
    handled separately by `predict_with_tta`, not by this flag).
    """

    def __init__(
        self, df, vocab, max_prompt_len=MAX_PROMPT_LEN, max_opt_len=MAX_OPT_LEN, has_labels=True, shuffle_options=False
    ):
        self.df = df.reset_index(drop=True)
        self.vocab = vocab
        self.max_prompt_len = max_prompt_len
        self.max_opt_len = max_opt_len
        self.has_labels = has_labels
        self.shuffle_options = shuffle_options

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt_ids = self.vocab.encode(row["prompt"], self.max_prompt_len)
        option_texts = [row[c] for c in OPTION_COLS]
        label_idx = LABEL2IDX[row["answer"]] if self.has_labels else None

        if self.shuffle_options:
            perm = list(range(5))
            random.shuffle(perm)
            option_texts = [option_texts[p] for p in perm]
            if label_idx is not None:
                label_idx = perm.index(label_idx)

        option_ids = [self.vocab.encode(t, self.max_opt_len) for t in option_texts]
        item = {
            "prompt_ids": torch.tensor(prompt_ids, dtype=torch.long),
            "option_ids": torch.tensor(option_ids, dtype=torch.long),  # [5, max_opt_len]
        }
        if self.has_labels:
            item["label"] = torch.tensor(label_idx, dtype=torch.long)
        return item


def lstm_collate(batch):
    out = {
        "prompt_ids": torch.stack([b["prompt_ids"] for b in batch]),
        "option_ids": torch.stack([b["option_ids"] for b in batch]),
    }
    if "label" in batch[0]:
        out["label"] = torch.stack([b["label"] for b in batch])
    return out


# ============================================================================
# MODEL
# ============================================================================

class LSTMEncoder(nn.Module):
    """Shared bidirectional LSTM encoder, mean-pooled over non-pad timesteps
    (more robust than the final hidden state when sequences are padded)."""

    def __init__(
        self, vocab_size, embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM,
        num_layers=NUM_LSTM_LAYERS, dropout=0.3, pad_id=0
    ):
        super().__init__()
        self.pad_id = pad_id
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, num_layers=num_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        mask = (x != self.pad_id).float()  # [batch, seq_len]
        emb = self.embedding(x)
        out, _ = self.lstm(emb)  # [batch, seq_len, hidden*2]
        lengths = mask.sum(dim=1, keepdim=True).clamp(min=1)
        pooled = (out * mask.unsqueeze(-1)).sum(dim=1) / lengths
        return self.dropout(pooled)  # [batch, hidden*2]


class MCQLstmModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM, pad_id=0):
        super().__init__()
        self.encoder = LSTMEncoder(vocab_size, embed_dim, hidden_dim, pad_id=pad_id)
        enc_dim = hidden_dim * 2
        self.scorer = nn.Sequential(
            nn.Linear(enc_dim * 4, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, prompt_ids, option_ids):
        # prompt_ids: [batch, plen] | option_ids: [batch, 5, olen]
        batch_size, n_opts, olen = option_ids.shape
        prompt_vec = self.encoder(prompt_ids)  # [batch, enc_dim]
        opt_vec = self.encoder(option_ids.view(batch_size * n_opts, olen))
        opt_vec = opt_vec.view(batch_size, n_opts, -1)  # [batch, 5, enc_dim]

        prompt_exp = prompt_vec.unsqueeze(1).expand(-1, n_opts, -1)  # [batch, 5, enc_dim]
        combined = torch.cat([
            prompt_exp, opt_vec,
            torch.abs(prompt_exp - opt_vec),
            prompt_exp * opt_vec,
        ], dim=-1)  # [batch, 5, enc_dim*4]

        return self.scorer(combined).squeeze(-1)  # [batch, 5] logits


# ============================================================================
# ERROR ANALYSIS — Macro F1 + confusion matrix
# ============================================================================

def run_error_analysis(df, probs, tag="lstm", log_to_wandb=False, save_plot=True):
    from sklearn.metrics import f1_score, confusion_matrix

    top1_pred_idx = np.argmax(probs, axis=1)
    pred_labels = [IDX2LABEL[i] for i in top1_pred_idx]
    true_labels = df["answer"].tolist()

    macro_f1 = f1_score(true_labels, pred_labels, average="macro", labels=OPTION_COLS)
    cm = confusion_matrix(true_labels, pred_labels, labels=OPTION_COLS)

    wrong_mask = np.array(pred_labels) != np.array(true_labels)
    wrong_examples = df.loc[wrong_mask, ["id", "prompt", "answer"]].copy()
    wrong_examples["predicted"] = np.array(pred_labels)[wrong_mask]
    wrong_examples = wrong_examples.head(20)

    cm_path = os.path.join(OUTPUT_DIR, f"{tag}_confusion_matrix.png")
    if save_plot:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(5, 4))
        im = ax.imshow(cm, cmap="Blues")
        ax.set_xticks(range(len(OPTION_COLS)))
        ax.set_yticks(range(len(OPTION_COLS)))
        ax.set_xticklabels(OPTION_COLS)
        ax.set_yticklabels(OPTION_COLS)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        ax.set_title(f"Confusion Matrix — {tag}")
        for i in range(len(OPTION_COLS)):
            for j in range(len(OPTION_COLS)):
                ax.text(j, i, cm[i, j], ha="center", va="center",
                        color="white" if cm[i, j] > cm.max() / 2 else "black")
        fig.colorbar(im)
        fig.tight_layout()
        fig.savefig(cm_path, dpi=150)
        plt.close(fig)

    if log_to_wandb:
        import wandb
        wandb.log({
            "macro_f1": macro_f1,
            "confusion_matrix_image": wandb.Image(cm_path) if save_plot else None,
            "misclassified_examples": wandb.Table(dataframe=wrong_examples),
        })

    return {"macro_f1": macro_f1, "confusion_matrix": cm, "wrong_examples": wrong_examples, "cm_path": cm_path}


# ============================================================================
# TRAIN / EVAL HELPERS
# ============================================================================

def run_eval(model, loader, device):
    """Returns (probs [n,5], labels list or []) — probs are softmax'd, not raw logits."""
    model.eval()
    probs_list, labels_list = [], []
    with torch.no_grad():
        for batch in loader:
            prompt_ids = batch["prompt_ids"].to(device)
            option_ids = batch["option_ids"].to(device)
            logits = model(prompt_ids, option_ids)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            probs_list.append(probs)
            if "label" in batch:
                labels_list.extend(batch["label"].numpy().tolist())
    return np.concatenate(probs_list, axis=0), labels_list


def predict_with_tta(
    model, df, vocab, device, n_augments=TTA_AUGMENTS, max_prompt_len=MAX_PROMPT_LEN, max_opt_len=MAX_OPT_LEN, batch_size=BATCH_SIZE
):
    """Test-time augmentation: run each row through the model with several random
    re-orderings of the 5 options, then average probabilities back in the original
    A-E order. `n_augments=1` just runs the original order (no augmentation) as a
    sanity-check baseline.
    """
    model.eval()
    ds = MCQLstmDataset(df, vocab, max_prompt_len, max_opt_len, has_labels=False, shuffle_options=False)
    n = len(df)
    prompt_tensor = torch.stack([ds[i]["prompt_ids"] for i in range(n)])
    option_tensor = torch.stack([ds[i]["option_ids"] for i in range(n)])  # [n, 5, olen]
    accumulated = np.zeros((n, 5), dtype="float64")

    for aug in range(n_augments):
        perm = np.arange(5) if aug == 0 else np.random.permutation(5)
        permuted_options = option_tensor[:, perm, :]
        probs_chunks = []

        with torch.no_grad():
            for start in range(0, n, batch_size):
                end = start + batch_size
                p_ids = prompt_tensor[start:end].to(device)
                o_ids = permuted_options[start:end].to(device)
                logits = model(p_ids, o_ids)
                probs_chunks.append(torch.softmax(logits, dim=1).cpu().numpy())

        probs_permuted = np.concatenate(probs_chunks, axis=0)  # [n, 5] in permuted order
        # un-permute: column i of probs_permuted is option that was originally at perm[i]
        unpermuted = np.zeros_like(probs_permuted)
        unpermuted[:, perm] = probs_permuted
        accumulated += unpermuted

    return accumulated / n_augments


def train_one_fold(tr_df, val_df, vocab, device, fold_idx, run_name):
    """Trains one LSTM on (tr_df, val_df). Returns (model, val_probs, val_map3)."""
    train_ds = MCQLstmDataset(tr_df, vocab, shuffle_options=SHUFFLE_OPTIONS_TRAINING)
    val_ds = MCQLstmDataset(val_df, vocab, shuffle_options=False)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=lstm_collate)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=lstm_collate)

    model = MCQLstmModel(len(vocab), pad_id=vocab.pad_id).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=1)
    criterion = nn.CrossEntropyLoss()

    if USE_WANDB:
        import wandb
        wandb.init(
            project=WANDB_PROJECT, entity=WANDB_ENTITY, name=run_name, reinit=True,
            config={
                "fold": fold_idx, "embed_dim": EMBED_DIM, "hidden_dim": HIDDEN_DIM, "lr": LR,
                "num_epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE, "vocab_size": len(vocab),
                "shuffle_options_training": SHUFFLE_OPTIONS_TRAINING,
            },
        )

    best_val_map3 = -1.0
    best_state = None
    epochs_no_improve = 0

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            optimizer.zero_grad()
            prompt_ids = batch["prompt_ids"].to(device)
            option_ids = batch["option_ids"].to(device)
            labels = batch["label"].to(device)

            logits = model(prompt_ids, option_ids)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)  # LSTMs can explode-gradient
            optimizer.step()
            total_loss += loss.item() * prompt_ids.size(0)

        train_loss = total_loss / len(train_ds)

        val_probs, val_labels_idx = run_eval(model, val_loader, device)
        val_true = [IDX2LABEL[l] for l in val_labels_idx]
        val_rankings = probs_to_top3(val_probs)
        val_map3 = mapk_score(val_true, val_rankings)
        scheduler.step(val_map3)

        print(f" [fold {fold_idx}] epoch {epoch:02d} | train_loss={train_loss:.4f} | val_map3={val_map3:.4f}")
        if USE_WANDB:
            import wandb
            wandb.log({
                "epoch": epoch, "train_loss": train_loss, "val_map3": val_map3,
                "lr": optimizer.param_groups[0]["lr"],
            })

        if val_map3 > best_val_map3:
            best_val_map3 = val_map3
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f" [fold {fold_idx}] early stopping at epoch {epoch}")
                break

    model.load_state_dict(best_state)
    model.to(device)

    final_val_probs, _ = run_eval(model, val_loader, device)
    if USE_WANDB:
        import wandb
        error_metrics = run_error_analysis(val_df, final_val_probs, tag=f"fold{fold_idx}", log_to_wandb=True)
        wandb.log({"final_val_map3": best_val_map3, "final_val_macro_f1": error_metrics["macro_f1"]})
        wandb.finish()

    return model, final_val_probs, best_val_map3


def train_lstm_kfold(train_df, test_df, n_folds=N_FOLDS):
    """K-fold CV + bagged test predictions + TTA.

    Returns:
        oof_probs : [len(train_df), 5] out-of-fold probabilities (each row predicted by
            the model that did NOT see it in training) — this gives a trustworthy
            overall validation score, unlike a single train/val split.
        oof_map3 : mAP@3 computed on oof_probs vs true answers
        test_probs_bagged : [len(test_df), 5] averaged across all fold models, each
            with TTA applied — this is what goes into the final submission.
        fold_models : list of trained models (in case you want to inspect per-fold behavior)
    """
    from sklearn.model_selection import StratifiedKFold

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Training on device: {device}")

    # vocab from train+test text (word surface forms only, no label leakage)
    vocab_texts = []
    for df in (train_df, test_df):
        for c in ["prompt"] + OPTION_COLS:
            vocab_texts.extend(df[c].tolist())
    vocab = Vocab(vocab_texts)
    print(f"Vocab size: {len(vocab)}")

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)

    oof_probs = np.zeros((len(train_df), 5))
    test_probs_sum = np.zeros((len(test_df), 5))
    fold_models = []
    fold_val_map3s = []

    for fold_idx, (tr_idx, val_idx) in enumerate(skf.split(train_df, train_df["answer"]), start=1):
        print(f"\n=== Fold {fold_idx}/{n_folds} ===")
        tr_df = train_df.iloc[tr_idx]
        val_df = train_df.iloc[val_idx]

        model, val_probs, val_map3 = train_one_fold(
            tr_df, val_df, vocab, device, fold_idx, run_name=f"lstm-fold{fold_idx}-of-{n_folds}"
        )
        oof_probs[val_idx] = val_probs
        fold_val_map3s.append(val_map3)
        fold_models.append(model)

        # bagged + TTA test predictions from this fold's model
        fold_test_probs = predict_with_tta(model, test_df, vocab, device, n_augments=TTA_AUGMENTS)
        test_probs_sum += fold_test_probs

    test_probs_bagged = test_probs_sum / n_folds

    oof_true = train_df["answer"].tolist()
    oof_rankings = probs_to_top3(oof_probs)
    oof_map3 = mapk_score(oof_true, oof_rankings)

    print(f"\nPer-fold val mAP@3: {[round(s, 4) for s in fold_val_map3s]}")
    print(f"Out-of-fold (overall) mAP@3: {oof_map3:.4f} <- trust this more than any single fold's number")

    return oof_probs, oof_map3, test_probs_bagged, fold_models, vocab


def build_submission(test_df, rankings, out_path=SUBMISSION_PATH):
    sub = pd.DataFrame({
        "ID": test_df["id"].values,
        "Prediction": [" ".join(r[:3]) for r in rankings],
    })
    sub.to_csv(out_path, index=False)
    print(f"Saved submission to {out_path}")
    return sub


# ============================================================================
# MAIN
# ============================================================================

def main():
    train_df, test_df = load_data()
    train_df = handle_missing_data(train_df, ["prompt"] + OPTION_COLS)
    test_df = handle_missing_data(test_df, ["prompt"] + OPTION_COLS)
    run_diagnostics(train_df)

    oof_probs, oof_map3, test_probs_bagged, fold_models, vocab = train_lstm_kfold(train_df, test_df)

    oof_error_metrics = run_error_analysis(train_df, oof_probs, tag="oof_overall", log_to_wandb=False)
    print(f"OOF Macro F1: {oof_error_metrics['macro_f1']:.4f}")

    final_rankings = probs_to_top3(test_probs_bagged)
    build_submission(test_df, final_rankings)


if __name__ == "__main__":
    main()



--- Diagnostics ---
Answer letter distribution:
answer
B    0.245
C    0.230
A    0.184
D    0.179
E    0.162
Name: proportion, dtype: float64
Duplicate prompts: 242 | Fully duplicate rows: 183
Avg length of CORRECT option: 181.9
Avg length per option column: {'A': np.float64(164.0915), 'B': np.float64(166.6165), 'C': np.float64(167.227), 'D': np.float64(162.5635), 'E': np.float64(164.23)}
--- End diagnostics ---

Training on device: cuda
Vocab size: 3004

=== Fold 1/5 ===


wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


...

epoch,▁▂▃▅▆▇█
final_val_macro_f1,▁
final_val_map3,▁
lr,████▃▃▁
macro_f1,▁
train_loss,█▃▁▁▁▁▁
val_map3,▁██████
epoch,7
final_val_macro_f1,0.94242
final_val_map3,0.96708
lr,0.00025



=== Fold 5/5 ===


 [fold 5] epoch 01 | train_loss=1.4486 | val_map3=0.8004
 [fold 5] epoch 02 | train_loss=0.4540 | val_map3=0.9513
 [fold 5] epoch 03 | train_loss=0.1867 | val_map3=0.9587
 [fold 5] epoch 04 | train_loss=0.1391 | val_map3=0.9537
 [fold 5] epoch 05 | train_loss=0.1201 | val_map3=0.9596
 [fold 5] epoch 06 | train_loss=0.1142 | val_map3=0.9596
 [fold 5] epoch 07 | train_loss=0.1060 | val_map3=0.9571
 [fold 5] epoch 08 | train_loss=0.1062 | val_map3=0.9583
 [fold 5] epoch 09 | train_loss=0.0967 | val_map3=0.9583
 [fold 5] early stopping at epoch 9


epoch,▁▂▃▄▅▅▆▇█
final_val_macro_f1,▁
final_val_map3,▁
lr,██████▃▃▁
macro_f1,▁
train_loss,█▃▁▁▁▁▁▁▁
val_map3,▁████████
epoch,9
final_val_macro_f1,0.93978
final_val_map3,0.95958
lr,0.00025



Per-fold val mAP@3: [0.9642, 0.96, 0.9658, 0.9671, 0.9596]
Out-of-fold (overall) mAP@3: 0.9633 <- trust this more than any single fold's number
OOF Macro F1: 0.9396
Saved submission to /content/drive/MyDrive/smart-mcq-solver/output_lstm_5vfold/submission_lstm_5v.csv


In [ ]:
# import os

# # Uninstall existing torch, torchvision, and torchaudio installations
# !pip uninstall -y torch torchvision torchaudio

# # Install the desired versions with CUDA 12.1 compatibility
# # Note: This command assumes a CUDA 12.1 environment. If your environment uses a different CUDA version,
# # you might need to adjust 'cu121' accordingly (e.g., 'cu118' for CUDA 11.8).
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 --force-reinstall

# # Restart the runtime to apply the changes
# os.kill(os.getpid(), 9)

Found existing installation: torch 2.11.0+cpu
Uninstalling torch-2.11.0+cpu:
  Successfully uninstalled torch-2.11.0+cpu
Found existing installation: torchvision 0.26.0+cpu
Uninstalling torchvision-0.26.0+cpu:
  Successfully uninstalled torchvision-0.26.0+cpu
Found existing installation: torchaudio 2.11.0+cpu
Uninstalling torchaudio-2.11.0+cpu:
  Successfully uninstalled torchaudio-2.11.0+cpu
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 25.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 67.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 63.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 38.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 84.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2